1: Import needed modules and functions

In [21]:
#used to setup the path to the workflow_helpers module
from pathlib import Path
import sys

#adds the notebooks directory to the path so that workflow_helpers can be imported
CANDIDATE_ROOTS = [Path.cwd().resolve(), Path.cwd().resolve().parent]
REPO_ROOT = next(path for path in CANDIDATE_ROOTS if (path / "notebooks" / "workflow_helpers").exists())
NOTEBOOKS_ROOT = REPO_ROOT / "notebooks"
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))

#imports the needed functions from the workflow_helpers module
from workflow_helpers.qiskit_uqk_helpers import (
    load_workflow_manifest,
    build_grouped_evolution_circuits,
    run_uqk_overlap_driver,
    solve_uqk_gep_from_manifest
)

# used to handle data
import numpy as np

# used to plot benchmarks
from matplotlib import pyplot as plt

# used to run the circuits on IBM Quantum
from qiskit_ibm_runtime import QiskitRuntimeService


2: Define a function to gather results from the quantum krylov procedure

In [ ]:

def do_qk(mol="diatomic_h2_sto_3g", sim="noiseless", kry_dim=4, dt=0.1, shots=20000, stochastic=False, nd=50, stochastic_instances=200):

    #load the manifest for the molecule and get the path to the manifest file
    MOLECULE_NAME = mol
    manifest_path, manifest = load_workflow_manifest(NOTEBOOKS_ROOT, MOLECULE_NAME)

    REBUILD_GROUPED_CIRCUITS = True
    DT=dt #IMPORTANT
    EVOLUTION_METHOD = "pauli_evolution_gate"  # "pauli_evolution_gate" or "manual_ladder"
    TARGET_MODE = "generic_simulator"          # "generic_simulator" or "ibm_qpu_paid"
    GENERIC_BASIS_GATES = ["rz", "sx", "x", "cx"]
    TRANSPILER_OPTIMIZATION_LEVEL = 3
    TROTTER_SEQUENCE_ORDER = 1
    IBM_BACKEND_NAME = "ibm_fez"
    IBM_RUNTIME_CHANNEL = "ibm_quantum_platform"
    IBM_INSTANCE = None
    IBM_USE_FRACTIONAL_GATES = False

    # --- User-facing UQK/MFE options ---
    if stochastic:
        UQK_MODE = "stochastic"
    else:
        UQK_MODE = "standard"

    if sim == "noiseless":
        BACKEND_MODE = "local_noiseless_statevector" # ibm AER?
    elif sim == "noisy-ibm":
        BACKEND_MODE = "local_noisy_ibm_model" # noisemodel tuned to ibm backend
    elif sim == "noisy-custom":
        BACKEND_MODE = "local_noisy_simple" # custom noisemodel

    KRYLOV_DIMENSION = kry_dim #IMPORTANT
    MAX_CORRELATION_POWER = KRYLOV_DIMENSION  # C_M is needed later for projected U.
    SHOTS_PER_MFE_EXPERIMENT = shots
    RANDOM_SEED = 230623
    MFE_VERBOSE_FOR_FIRST_NONZERO_POWER = False

    # --- User-facing stochastic qDRIFT options ---
    QDRIFT_SEGMENT_COUNT_ND = nd
    STOCHASTIC_INSTANCES_PER_CORRELATION = stochastic_instances
    STOCHASTIC_WEIGHT_CONVENTION = "group_pauli_l1_norm"

    # --- User-facing noisy simulator options ---
    NOISY_SIMULATION_METHOD = "density_matrix"
    NOISY_TRANSPILE_OPTIMIZATION_LEVEL = 1
    SIMPLE_NOISE_ONE_QUBIT_DEPOLARIZING_PROBABILITY = 1.0e-3
    SIMPLE_NOISE_TWO_QUBIT_DEPOLARIZING_PROBABILITY = 1.0e-2
    SIMPLE_NOISE_READOUT_ERROR_PROBABILITY = 2.0e-2
    IBM_MODEL_SOURCE = "fake_backend"
    IBM_MODEL_FAKE_BACKEND_CLASS = "FakeBrisbane"
    IBM_MODEL_RUNTIME_BACKEND_NAME = "ibm_brisbane"
    IBM_MODEL_COMPRESS_TO_ACTIVE_SPACE = True

    OUTPUT_LABEL = f"{MOLECULE_NAME}_{UQK_MODE}_{BACKEND_MODE}_{SHOTS_PER_MFE_EXPERIMENT}_shots"

    if REBUILD_GROUPED_CIRCUITS:
        manifest, circuit_metadata = build_grouped_evolution_circuits(
            manifest_path=manifest_path,
            manifest=manifest,
            dt=DT,
            evolution_method=EVOLUTION_METHOD,
            target_mode=TARGET_MODE,
            generic_basis_gates=GENERIC_BASIS_GATES,
            transpiler_optimization_level=TRANSPILER_OPTIMIZATION_LEVEL,
            trotter_sequence_order=TROTTER_SEQUENCE_ORDER,
            ibm_backend_name=IBM_BACKEND_NAME,
            ibm_runtime_channel=IBM_RUNTIME_CHANNEL,
            ibm_instance=IBM_INSTANCE,
            ibm_use_fractional_gates=IBM_USE_FRACTIONAL_GATES,
            verbose=False,
        )

    manifest, uqk_data, uqk_metadata = run_uqk_overlap_driver(
        manifest_path=manifest_path,
        manifest=manifest,
        uqk_mode=UQK_MODE,
        backend_mode=BACKEND_MODE,
        krylov_dimension=KRYLOV_DIMENSION,
        max_correlation_power=MAX_CORRELATION_POWER,
        shots_per_mfe_experiment=SHOTS_PER_MFE_EXPERIMENT,
        qdrift_segment_count_Nd=QDRIFT_SEGMENT_COUNT_ND,
        stochastic_instances_per_correlation=STOCHASTIC_INSTANCES_PER_CORRELATION,
        stochastic_weight_convention=STOCHASTIC_WEIGHT_CONVENTION,
        random_seed=RANDOM_SEED,
        output_label=OUTPUT_LABEL,
        noisy_simulation_method=NOISY_SIMULATION_METHOD,
        noisy_transpile_optimization_level=NOISY_TRANSPILE_OPTIMIZATION_LEVEL,
        simple_noise_one_qubit_depolarizing_probability=SIMPLE_NOISE_ONE_QUBIT_DEPOLARIZING_PROBABILITY,
        simple_noise_two_qubit_depolarizing_probability=SIMPLE_NOISE_TWO_QUBIT_DEPOLARIZING_PROBABILITY,
        simple_noise_readout_error_probability=SIMPLE_NOISE_READOUT_ERROR_PROBABILITY,
        ibm_model_source=IBM_MODEL_SOURCE,
        ibm_model_fake_backend_class=IBM_MODEL_FAKE_BACKEND_CLASS,
        ibm_model_runtime_backend_name=IBM_MODEL_RUNTIME_BACKEND_NAME,
        ibm_model_compress_to_active_space=IBM_MODEL_COMPRESS_TO_ACTIVE_SPACE,
        mfe_verbose_for_first_nonzero_power=MFE_VERBOSE_FOR_FIRST_NONZERO_POWER,
    )

    # --- User-facing GEP options ---
    KRYLOV_DIMENSION_TO_USE = KRYLOV_DIMENSION
    KRYLOV_DT = DT
    OVERLAP_EIGENVALUE_THRESHOLD = None  # None mirrors qforte's default 1e-15 behavior.
    SORT_ROOTS_BY = "energy_real_ascending"
    USE_DIRECT_VALIDATION_WHEN_AVAILABLE = True
    OUTPUT_SUMMARY_NAME = f"{OUTPUT_LABEL}_uqk_gep_summary.json"

    manifest, gep_summary = solve_uqk_gep_from_manifest(
        manifest_path=manifest_path,
        manifest=manifest,
        krylov_dimension_to_use=KRYLOV_DIMENSION_TO_USE,
        dt=KRYLOV_DT,
        overlap_eigenvalue_threshold=OVERLAP_EIGENVALUE_THRESHOLD,
        sort_roots_by=SORT_ROOTS_BY,
        output_summary_name=OUTPUT_SUMMARY_NAME,
        use_direct_validation_when_available=USE_DIRECT_VALIDATION_WHEN_AVAILABLE,
    )

    #return gep_summary["input_correlation_solution"]["energies_hartree"][0]["energy"]
    return gep_summary, uqk_data

: 

3: Define a function for computing batches of test instances

In [32]:
def benchmark(trials = 5, mol="diatomic_h2_sto_3g", sim="noisy-ibm", kry_dim=[5], dt=[0.1], shots=[2000, 25000], nd=[1, 1000], stochastic_instances=[50, 5000]):

    results = []

    # run the requested number of trials and compute the absolute and relative frobenius norm errors between the standard and stochastic UQK results
    for trial in range(trials):

        # set up trial parameters
        t_kry_dim = kry_dim[0] + trial * (kry_dim[-1] - kry_dim[0]) // trials
        t_dt = dt[0] + trial * (dt[-1] - dt[0]) // trials
        t_shots = shots[0] + trial * (shots[-1] - shots[0]) // trials
        t_nd = nd[0] + trial * (nd[-1] - nd[0]) // trials
        t_stochastic_instances = stochastic_instances[0] + trial * (stochastic_instances[-1] - stochastic_instances[0]) // trials

        # run the standard and stochastic UQK calculations
        print(f"Running trial {trial + 1}/{trials} with parameters: kry_dim={t_kry_dim}, dt={t_dt}, shots={t_shots}, nd={t_nd}, stochastic_instances={t_stochastic_instances}")
        result_std, data_std = do_qk(mol=mol, sim=sim, kry_dim=t_kry_dim, dt=t_dt, shots=t_shots, stochastic=False, nd=t_nd, stochastic_instances=t_stochastic_instances)
        result_stoch, data_stoch = do_qk(mol=mol, sim=sim, kry_dim=t_kry_dim, dt=t_dt, shots=t_shots, stochastic=True, nd=t_nd, stochastic_instances=t_stochastic_instances)

        # compute the absolute and relative frobenius norm errors between the standard and stochastic UQK results
        abs_frob_err = np.linalg.norm(data_stoch["S"] - data_std["S"])
        rel_frob_err = abs_frob_err / np.linalg.norm(data_std["S"])

        # store the results for this trial
        results.append({
            "trial": trial,
            "kry_dim": t_kry_dim,
            "dt": t_dt,
            "shots": t_shots,
            "nd": t_nd,
            "stochastic_instances": t_stochastic_instances,
            "result_std": result_std,
            "result_stoch": result_stoch,
            "abs_frob_err": abs_frob_err,
            "rel_frob_err": rel_frob_err
        })

    return results

In [35]:
results = benchmark(trials=5, mol="diatomic_h2_sto_3g", sim="noisy-ibm", kry_dim=[5], dt=[0.1], shots=[2000, 10000], nd=[1, 100], stochastic_instances=[50, 250])


Loaded Workflow Manifest
Molecule name:                           diatomic_h2_sto_3g
Manifest JSON:                           /home/zero/remote/cs-github/qforte-qiskit/notebooks/diatomic_h2_sto_3g/metadata/workflow_manifest.json
Workflow root:                           /home/zero/remote/cs-github/qforte-qiskit/notebooks/diatomic_h2_sto_3g
file: fermion_blocks_json                /home/zero/remote/cs-github/qforte-qiskit/notebooks/diatomic_h2_sto_3g/hamiltonian_blocks/diatomic_h2_sto_3g_fermion_blocks.json
file: grouped_evolution_metadata_json    /home/zero/remote/cs-github/qforte-qiskit/notebooks/diatomic_h2_sto_3g/circuits/diatomic_h2_sto_3g_grouped_evolution_metadata.json
file: grouped_evolution_qpy              /home/zero/remote/cs-github/qforte-qiskit/notebooks/diatomic_h2_sto_3g/circuits/diatomic_h2_sto_3g_grouped_evolution.qpy
file: grouped_paulis_json                /home/zero/remote/cs-github/qforte-qiskit/notebooks/diatomic_h2_sto_3g/hamiltonian_blocks/diatomic_h2_sto_3g_grou

KeyboardInterrupt: 

In [ ]:
def plot_benchmark_results(results):
    for result in results:
        trial = result["trial"]
        kry_dim = result["kry_dim"]
        dt = result["dt"]
        shots = result["shots"]
        nd = result["nd"]
        stochastic_instances = result["stochastic_instances"]
        abs_frob_err = result["abs_frob_err"]
        rel_frob_err = result["rel_frob_err"]

        print(f"Trial {trial}: kry_dim={kry_dim}, dt={dt}, shots={shots}, nd={nd}, stochastic_instances={stochastic_instances}, abs_frob_err={abs_frob_err:.6e}, rel_frob_err={rel_frob_err:.6e}")
    pass

In [ ]:
# Create (once) and then keep adding curves to the same axes
if "fig_bench" not in globals() or "axs_bench" not in globals():
    fig_bench, axs_bench = plt.subplots(1, 2, figsize=(12, 4))

    axs_bench[0].set_title("Relative Frobenius Error vs Trial")
    axs_bench[0].set_xlabel("Trial")
    axs_bench[0].set_ylabel("Relative Frobenius Error")
    axs_bench[0].grid(True, alpha=0.3)

    axs_bench[1].set_title("Absolute Frobenius Error vs Trial")
    axs_bench[1].set_xlabel("Trial")
    axs_bench[1].set_ylabel("Absolute Frobenius Error")
    axs_bench[1].grid(True, alpha=0.3)

def add_results_to_plots(results_to_add, label=None):
    trials = [r["trial"] for r in results_to_add]
    rel_err = [r["rel_frob_err"] for r in results_to_add]
    abs_err = [r["abs_frob_err"] for r in results_to_add]

    if label is None:
        label = f"run_{len(axs_bench[0].lines) + 1}"

    axs_bench[0].plot(trials, rel_err, marker="o", label=label)
    axs_bench[1].plot(trials, abs_err, marker="o", label=label)

    axs_bench[0].legend()
    axs_bench[1].legend()
    fig_bench.tight_layout()
    fig_bench.canvas.draw_idle()

# Add current data
add_results_to_plots(results, label="initial")

# Later, after generating more results, just call:
# more_results = benchmark(...)
# add_results_to_plots(more_results, label="more_data")